In [ ]:
import customtkinter as ctk
from tkinter import messagebox
import random
import logging
from datetime import datetime
from passlib.hash import pbkdf2_sha256
import time
import threading

logging.basicConfig(level=logging.INFO, format='%(asctime)s - [SHARKSHELL_SENTINEL] - %(message)s')
ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

class BlueFinBank(ctk.CTkToplevel):
    def __init__(self, parent):
        super().__init__(parent)
        self.parent = parent 
        self.title("Blue Fin Private Wealth - Secure Terminal")
        self.geometry("900x650")
        self.configure(fg_color="#0a192f")
        self.attributes("-topmost", True)

        self.sidebar = ctk.CTkFrame(self, width=200, fg_color="#0d253f")
        self.sidebar.pack(side="left", fill="y")

        ctk.CTkLabel(self.sidebar, text="BLUE FIN", font=("Arial", 22, "bold"), text_color="#3498db").pack(pady=30)

        for btn in ["Dashboard", "Accounts", "Transfers", "Investments"]:
            ctk.CTkButton(self.sidebar, text=btn, fg_color="transparent", anchor="w").pack(fill="x", padx=20, pady=5)

        self.content = ctk.CTkFrame(self, fg_color="transparent")
        self.content.pack(side="right", fill="both", expand=True, padx=30, pady=30)

        header = ctk.CTkFrame(self.content, fg_color="transparent")
        header.pack(fill="x", pady=(0, 20))

        ctk.CTkLabel(header, text=f"Welcome, {getattr(parent, 'current_user', 'User')}", font=("Arial", 24, "bold")).pack(side="left")
        ctk.CTkLabel(header, text=f"Node: MO-USA | {datetime.now().strftime('%H:%M')}", text_color="gray").pack(side="right")

        card = ctk.CTkFrame(self.content, fg_color="#1f6aa5", height=120, corner_radius=15)
        card.pack(fill="x", pady=10)
        card.pack_propagate(False)

        ctk.CTkLabel(card, text="TOTAL ASSETS", font=("Arial", 12, "bold")).pack(pady=(15, 0))
        ctk.CTkLabel(card, text="$1,245,800.50", font=("Arial", 36, "bold")).pack()

        ctk.CTkLabel(self.content, text="Recent Transactions", font=("Arial", 16, "bold")).pack(anchor="w", pady=(20, 10))

        ledger = ctk.CTkTextbox(self.content, height=250, fg_color="#0d253f", font=("Courier", 12))
        ledger.pack(fill="both", expand=True)

        log_data = [
            "2026-03-23 | Wire Transfer IN | +$15,000.00",
            "2026-03-22 | Server Hosting Fee | -$1,240.20",
            "2026-03-21 | SharkShell Premium | -$199.99",
            "2026-03-21 | Dividends Received | +$2,100.00"
        ] 
        ledger.insert("0.0", "\n".join(log_data))
        ledger.configure(state="disabled")

        
        ctk.CTkButton(self.content, text="SECURE TERMINAL LOGOUT", 
                      command=self.bank_exit, 
                      fg_color="#c0392b", hover_color="#a93226").pack(pady=20)

    def bank_exit(self):
        self.destroy() 
        self.parent.secure_logout() 

class SharkShellApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title("SharkShell Enterprise Security")
        self.geometry("750x850")
        self.configure(fg_color="#050a14")

        self._user_db = {
            "mark123": pbkdf2_sha256.hash("1234"),
            "admin": pbkdf2_sha256.hash("4321")
        }

        self.flagged_users = set()  
        self.activity_log = []
        self.honeypot_log = []
        self.failed_attempts = 0
        self.honeypot_enabled = True
        self.current_user = None

        self.main_container = ctk.CTkFrame(self, corner_radius=20, fg_color="#0a192f")
        self.main_container.pack(padx=40, pady=40, fill="both", expand=True)

        self.render_login()

    def secure_logout(self):
        """Universal logout method to scrub session and return home."""
        self.log_event(f"Session terminated for {self.current_user}")
        self.current_user = None
        self.failed_attempts = 0
        messagebox.showinfo("Security", "Session cleared. Returning to login...")
        self.render_login()

    def log_event(self, action):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        entry = {"action": action, "time": timestamp}
        self.activity_log.insert(0, entry)
        logging.info(f"ACTION LOGGED: {action}")
        if hasattr(self, 'hist_box') and self.hist_box.winfo_exists():
            self.update_history_display()

    def render_login(self):
        # UI Reset
        self.configure(fg_color="#050a14")
        self.main_container.configure(fg_color="#0a192f")
        for w in self.main_container.winfo_children():
            w.destroy()

        ctk.CTkLabel(self.main_container, text="🦈", font=("Arial", 60)).pack(pady=(60, 10))
        ctk.CTkLabel(self.main_container, text="SHARKSHELL", font=("Arial", 30, "bold"), text_color="#3498db").pack(pady=5)
        ctk.CTkLabel(self.main_container, text="ENTERPRISE AUTHENTICATION", font=("Arial", 12)).pack(pady=5)

        self.user_input = ctk.CTkEntry(self.main_container, placeholder_text="Username", width=300, height=45)
        self.user_input.pack(pady=(40, 10))

        self.pass_input = ctk.CTkEntry(self.main_container, placeholder_text="Password", show="*", width=300, height=45)
        self.pass_input.pack(pady=10)

        self.hp_field = ctk.CTkEntry(self.main_container, width=0, height=0)
        self.hp_field.place(x=-1000, y=-1000)

        ctk.CTkButton(self.main_container, text="AUTHORIZE SESSION", command=self.handle_login, width=300, height=50).pack(pady=40)

    def handle_login(self):
        u, p = self.user_input.get(), self.pass_input.get()

        if self.honeypot_enabled and self.hp_field.get():
            if u: self.flagged_users.add(u)
            self.trigger_deception()
            return

        hashed = self._user_db.get(u)
        if hashed and pbkdf2_sha256.verify(p, hashed):
            if u in self.flagged_users:
                self.render_verification_gate(u)
                return

            self.current_user = u
            self.log_event(f"{u} Authentication Success")
            self.show_hub()
        else:
            self.failed_attempts += 1
            self.log_event(f"Failed Login Attempt ({self.failed_attempts})")
            if self.failed_attempts >= 3:
                if u: self.flagged_users.add(u)
                self.trigger_deception()
            else:
                messagebox.showerror("Access Denied", "Credentials rejected by system.")

    def render_verification_gate(self, username):
        for w in self.main_container.winfo_children():
            w.destroy()

        ctk.CTkLabel(self.main_container, text="⚠️ SYSTEM INTEGRITY ALERT", font=("Arial", 24, "bold"), text_color="#e74c3c").pack(pady=20)
        ctk.CTkLabel(self.main_container, text=f"Access restricted for: {username}\nContact an administrator to clear your Node ID.", wraplength=400).pack(pady=10)

        ctk.CTkLabel(self.main_container, text="PRIMARY NODE ID").pack(pady=(20, 5))
        user_answer = ctk.CTkEntry(self.main_container, width=300)
        user_answer.pack(pady=5)

        ctk.CTkLabel(self.main_container, text="ADMIN OVERRIDE", font=("Arial", 12, "bold")).pack(pady=(30, 5))
        admin_pass = ctk.CTkEntry(self.main_container, show="*", width=300)
        admin_pass.pack(pady=5)

        def verify_bypass():
            a_pass = admin_pass.get()
            u_ans = user_answer.get()
            admin_hashed = self._user_db.get("admin")
            if pbkdf2_sha256.verify(a_pass, admin_hashed) and u_ans.strip():
                self.flagged_users.remove(username)
                self.log_event(f"Admin Restored Access for {username}")
                messagebox.showinfo("Success", "Security flag cleared. Proceed to Login.")
                self.render_login()
            else:
                messagebox.showerror("Critical Fail", "Administrative override rejected.") 

        ctk.CTkButton(self.main_container, text="UNLOCK ACCOUNT", command=verify_bypass, fg_color="#27ae60", width=300).pack(pady=30)
        ctk.CTkButton(self.main_container, text="Back to Login", command=self.render_login, fg_color="transparent").pack()

    def show_hub(self):
        for w in self.main_container.winfo_children():
            w.destroy()

        self.tabs = ctk.CTkTabview(self.main_container, width=650, height=650)
        self.tabs.pack(padx=20, pady=20)

        if self.current_user == "admin":
            self.tabs.add("Security Logs")
            self.tabs.add("Honeypot Events")
            self.tabs.add("Bank Oversight")

            self.hist_box = ctk.CTkTextbox(self.tabs.tab("Security Logs"), width=500, height=400, font=("Courier", 12))
            self.hist_box.pack(pady=10)
            self.update_history_display()

            self.hp_box = ctk.CTkTextbox(self.tabs.tab("Honeypot Events"), width=500, height=400, font=("Courier", 12))
            self.hp_box.pack(pady=10)
            self.update_honeypot_display()

            ctk.CTkLabel(self.tabs.tab("Bank Oversight"), text="CENTRAL BANKING OVERRIDE", font=("Arial", 18, "bold")).pack(pady=20)
            ctk.CTkButton(self.tabs.tab("Bank Oversight"), text="Access Blue Fin Node", command=lambda: BlueFinBank(self)).pack(pady=10)
        else:
            self.tabs.add("Services")
            self.tabs.add("History")
            self.tabs.add("Settings")

            ctk.CTkLabel(self.tabs.tab("Services"), text="AUTHORIZED CLOUD SERVICES", font=("Arial", 20, "bold")).pack(pady=20)
            nodes = [
                ("💰 Blue Fin Private Wealth", self.bank_mfa),
                ("☁️ Private Cloud Vault", lambda: self.launch_service("Cloud Vault")),
                ("🛋️ Furniture Biz Manager", lambda: self.launch_service("Inventory Manager")),
                ("🖼️ Photo Vault", lambda: self.launch_service("Media Gallery"))
            ]
            for name, cmd in nodes:
                ctk.CTkButton(self.tabs.tab("Services"), text=name, width=350, height=40, command=cmd).pack(pady=8)

            self.hist_box = ctk.CTkTextbox(self.tabs.tab("History"), width=500, height=400, font=("Courier", 12))
            self.hist_box.pack(pady=10)
            self.update_history_display()

            hp_switch = ctk.CTkSwitch(self.tabs.tab("Settings"), text="Honeypot Deception", command=self.toggle_hp)
            if self.honeypot_enabled: hp_switch.select()
            hp_switch.pack(pady=20)

        
        ctk.CTkButton(self.main_container, text="SECURE SIGN OUT", 
                      command=self.secure_logout, 
                      fg_color="#c0392b", hover_color="#a93226").pack(pady=10)

    def update_history_display(self):
        self.hist_box.configure(state="normal")
        self.hist_box.delete("0.0", "end")
        header = f"{'ACTION':<30} | {'TIMESTAMP':<20}\n" + "-" * 55 + "\n"
        self.hist_box.insert("end", header)
        for entry in self.activity_log:
            self.hist_box.insert("end", f"{entry['action']:<30} | {entry['time']:<20}\n")
        self.hist_box.configure(state="disabled")

    def update_honeypot_display(self):
        self.hp_box.configure(state="normal")
        self.hp_box.delete("0.0", "end")
        header = f"{'EVENT':<25} | {'TIMESTAMP':<20}\n" + "-" * 50 + "\n"
        self.hp_box.insert("end", header)
        for entry in self.honeypot_log:
            self.hp_box.insert("end", f"{entry['event']:<25} | {entry['time']:<20}\n")
        self.hp_box.configure(state="disabled")

    def launch_service(self, name):
        self.log_event(f"Accessed {name}")
        messagebox.showinfo(name, f"Establishing encrypted tunnel to {name}...")

    def toggle_hp(self):
        self.honeypot_enabled = not self.honeypot_enabled
        self.log_event(f"Honeypot status changed: {self.honeypot_enabled}")

    def bank_mfa(self):
        self.log_event("Blue Fin MFA Requested")
        otp = str(random.randint(100000, 999999))
        messagebox.showinfo("Secure MFA", f"System Token: {otp}")
        user_otp = ctk.CTkInputDialog(text="Enter 6-digit MFA token:", title="Auth").get_input()

        if user_otp == otp:
            self.log_event("Blue Fin Authorized")
            BlueFinBank(self)
        else:
            self.log_event("MFA Verification Failed")
            messagebox.showerror("Denied", "Invalid Token.")

    def trigger_deception(self):
        for w in self.main_container.winfo_children():
            w.destroy()
        self.configure(fg_color="#000000")
        self.main_container.configure(fg_color="#000000") 
        self.console = ctk.CTkTextbox(self.main_container, fg_color="#000000", text_color="#2ecc71", font=("Courier", 13), border_width=0)
        self.console.pack(fill="both", expand=True, padx=10, pady=10)
        threading.Thread(target=self.run_fake_terminal, daemon=True).start()

    def run_fake_terminal(self):
        def write(msg, delay=0.1):
            if self.console.winfo_exists():
                self.console.insert("end", msg + "\n")
                self.console.see("end")
                time.sleep(delay)

        write("> [!] CRITICAL: Kernel Overflow Exploited", 0.5)
        write("> Escalating to ROOT...", 0.4)
        write("> [OK] root@sharkshell:~# ", 0.8)
        write("> ls -la /mnt/secure_vault/", 0.2)
        write("drwxr-xr-x 2 root root 4096 offshore_accounts.db", 0.1)
        write("> decrypt private_keys.tar.gz --force", 0.5)
        for i in range(0, 101, 20):
            write(f"> BRUTE FORCING: [{'*'*(i//10)}{' '*(10-(i//10))}] {i}%", 0.4)
        write("> [SUCCESS] Accessing offshore_accounts.db...", 1.0)
        self.after(500, self.activate_delta_protocol)

    def activate_delta_protocol(self):
        self.configure(fg_color="#4b0000")
        self.main_container.configure(fg_color="#4b0000")
        self.console.configure(text_color="#ffffff", fg_color="#4b0000")
        self.console.insert("end", "\n" + "!" * 50 + "\n!!! TRAPWIRE ACTIVATED: SILENT ALARM !!!\n!!! INITIATING REMOTE WIPE... !!!\n" + "!" * 50 + "\n")
        self.honeypot_log.insert(0, {"event": "Intruder trapped", "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
        self.log_event("SHARKSHELL HONEYPOT: Intruder trapped.")
        self.after(5000, self.render_login)

if __name__ == "__main__":
    app = SharkShellApp()
    app.mainloop()

2026-04-13 12:54:25,798 - [SHARKSHELL_SENTINEL] - ACTION LOGGED: Failed Login Attempt (1)
2026-04-13 12:54:27,107 - [SHARKSHELL_SENTINEL] - ACTION LOGGED: Failed Login Attempt (2)
2026-04-13 12:54:28,147 - [SHARKSHELL_SENTINEL] - ACTION LOGGED: Failed Login Attempt (3)
2026-04-13 12:54:34,764 - [SHARKSHELL_SENTINEL] - ACTION LOGGED: SHARKSHELL HONEYPOT: Intruder trapped.
2026-04-13 12:55:04,808 - [SHARKSHELL_SENTINEL] - ACTION LOGGED: Admin Restored Access for mark123
2026-04-13 12:55:11,410 - [SHARKSHELL_SENTINEL] - ACTION LOGGED: admin Authentication Success
